<a href="https://colab.research.google.com/github/stsaganos/Machine-Learning-in-Finance/blob/main/Assignment_Week3_SPYRIDON_TSAGANOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Load Data and Document Source

**Dataset Documentation:**
* **Source & Citation:** "Default of Credit Card Clients" (Dataset 350) from the UCI Machine Learning Repository.
* **Observation Unit:** One credit-card account record per borrower.
* **Target Variable:** `default_next_month` (1 indicates default next month, 0 indicates no default).
* **Variable Meanings:**
  * ID: Unique borrower identifier (used for traceability only, excluded from predictors).
  * LIMIT_BAL: Credit limit in New Taiwan dollars (numeric).
  * Demographics: `SEX`, `EDUCATION`, and `MARRIAGE` (categorical codes supplied by the source).
  * AGE: Age in years (numeric).
  * Financial History: `PAY_0` to `PAY_6` (repayment status codes), `BILL_AMT1` to `BILL_AMT6` (bill statement amounts), and `PAY_AMT1` to `PAY_AMT6` (previous payment amounts).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

DATA_URL = ("https://raw.githubusercontent.com/fkottas/Teaching-Courses/main/"
    "Machine%20Learning%20in%20Finance/2026-2027/lectures/lecture-00-01/"
    "data/credit/default_of_credit_card_clients.csv.gz")
credit = pd.read_csv(DATA_URL, compression="gzip")
credit = credit.rename(columns={"default payment next month": "default_next_month"})

2. Data Audit: We inspect the dataset to verify its shape, data types, missingness, duplicate row count, unique-ID count, target balance, and coded-category frequencies.

In [2]:
print("shape:", credit.shape)
print(credit.head(3).to_string(index=False))

audit = pd.DataFrame({
    "dtype": credit.dtypes.astype(str),
    "missing": credit.isna().sum(),
    "unique": credit.nunique(),
})
print(audit.head(10).to_string())
print("\nexact duplicate rows:", int(credit.duplicated().sum()))
print("unique IDs:", int(credit["ID"].nunique()))

shape: (30000, 25)
 ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default_next_month
  1      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0                   1
  2     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000                   1
  3      90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948      15549      1518      1500      1000      1000      1000      5000                   0
           dtype  missing  unique
ID         

In [3]:
print("target counts:")
print(credit["default_next_month"].value_counts().sort_index())
print("\nEDUCATION codes:")
print(credit["EDUCATION"].value_counts().sort_index())
print("\nSEX codes:")
print(credit["SEX"].value_counts().sort_index())
print("\nMARRIAGE codes:")
print(credit["MARRIAGE"].value_counts().sort_index())

target counts:
default_next_month
0    23364
1     6636
Name: count, dtype: int64

EDUCATION codes:
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

SEX codes:
SEX
1    11888
2    18112
Name: count, dtype: int64

MARRIAGE codes:
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64


3. Controlled Missingness Example: We create a small in-memory dataset to demonstrate missing value treatment. We use a combination of an indicator (flag) and median imputation. Imputation allows the numeric transformer to process the column without crashing, while the indicator preserves the behavioral fact that the original source did not report this value (which could be economically significant). We avoid row removal to prevent data loss.

In [4]:
demo = credit[["LIMIT_BAL", "AGE", "EDUCATION", "MARRIAGE"]].head(10).copy()
demo.loc[[1, 2, 7], "AGE"] = np.nan
demo.loc[[4, 6], "EDUCATION"] = np.nan
missing_rate = demo.isna().mean()
print(missing_rate.to_string())
demo["AGE_was_missing"] = demo["AGE"].isna().astype(int)
demo["AGE"] = demo["AGE"].fillna(demo["AGE"].median())
print("\nAge after imputation and missingness flag:")
print(demo[["AGE", "AGE_was_missing"]].to_string(index=False))


LIMIT_BAL    0.0
AGE          0.3
EDUCATION    0.2
MARRIAGE     0.0

Age after imputation and missingness flag:
 AGE  AGE_was_missing
24.0                0
35.0                1
35.0                1
37.0                0
57.0                0
37.0                0
29.0                0
35.0                1
28.0                0
35.0                0


4. Duplicate Detection Example: We construct a small dataset with duplicate records to demonstrate identifier validation. An identical borrower ID is not automatically evidence that two rows represent distinct economic entities. In financial datasets, multiple rows with the same ID may represent repeated transactions over time, distinct financial products held by the same borrower, or system ingestion logs rather than duplicate individuals.

In [5]:
duplicate_demo = pd.concat([credit.head(10), credit.head(4)], ignore_index=True)
print("rows:", len(duplicate_demo))
print("exact duplicates:", int(duplicate_demo.duplicated().sum()))
print("duplicated IDs:", int(duplicate_demo["ID"].duplicated().sum()))

rows: 14
exact duplicates: 4
duplicated IDs: 4


5. Outliers: Flag First, Decide Later: A large bill statement amount (or negative balance) may be valid. We use the 1st and 99th percentiles of `BILL_AMT1` for a transparent teaching flag. The flag does not delete observations and is not a claim that the rows are errors. In financial modeling, extreme balances often represent real economic signals (e.g., credit balances or high credit utilization) rather than corrupt data.

In [6]:
lower, upper = credit["BILL_AMT1"].quantile([0.01, 0.99])
limit_flag = ~credit["BILL_AMT1"].between(lower, upper)
print("lower threshold:", lower)
print("upper threshold:", upper)
print("within 1%-99%:", int((~limit_flag).sum()))
print("outside 1%-99%:", int(limit_flag.sum()))

lower threshold: -81.0
upper threshold: 350110.6799999999
within 1%-99%: 29401
outside 1%-99%: 599


In [7]:
outlier_summary = pd.DataFrame({
    "variable": ["LIMIT_BAL", "AGE", "BILL_AMT1", "PAY_AMT1"],
    "q1": [50000, 28, 3558.75, 1000],
    "median": [140000, 34, 22381.5, 2100],
    "q3": [240000, 41, 67091, 5006],
})
print(outlier_summary.to_string(index=False))

 variable       q1   median     q3
LIMIT_BAL 50000.00 140000.0 240000
      AGE    28.00     34.0     41
BILL_AMT1  3558.75  22381.5  67091
 PAY_AMT1  1000.00   2100.0   5006


6. Separate variable types before encoding: The teaching block uses five numeric predictors and three categorical predictors. The identifier remains available for traceability but is excluded from X; the target is kept in y.

In [8]:
num_cols = ["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1", "PAY_AMT1"]
cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]
X = credit[num_cols + cat_cols]
y = credit["default_next_month"]
print("numeric:", num_cols)
print("categorical:", cat_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)

numeric: ['LIMIT_BAL', 'AGE', 'PAY_0', 'BILL_AMT1', 'PAY_AMT1']
categorical: ['SEX', 'EDUCATION', 'MARRIAGE']
X shape: (30000, 8)
y shape: (30000,)


In [9]:
cats = credit[["SEX", "EDUCATION", "MARRIAGE"]].head()
encoded = pd.get_dummies(cats.astype("string"), drop_first=False)
print(encoded.shape)
print(encoded.columns.tolist())

(5, 5)
['SEX_1', 'SEX_2', 'EDUCATION_2', 'MARRIAGE_1', 'MARRIAGE_2']


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train default rate:", round(y_train.mean(), 4))
print("y_test default rate:", round(y_test.mean(), 4))

X_train: (24000, 8)
X_test: (6000, 8)
y_train default rate: 0.2212
y_test default rate: 0.2212


7. Build One Leakage-Safe ColumnTransformer: We create a unified preprocessing pipeline to handle numeric and categorical features simultaneously without data leakage.

 - **Numeric Features**: Processed with median imputation for missing values and standard scaling ($\mu=0, \sigma=1$).
 - **Categorical Features**: Processed with most-frequent imputation and One-Hot Encoding (`handle_unknown="ignore"`).

The transformer is fitted strictly on `X_train` (`fit_transform`), while `X_test` is only transformed (`transform`), guaranteeing zero information leakage from the test split.

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train[["LIMIT_BAL", "PAY_AMT1"]])
test_scaled = scaler.transform(X_test[["LIMIT_BAL", "PAY_AMT1"]])
print("Scaled shapes (Train / Test):", train_scaled.shape, test_scaled.shape)
print("Training means (should be ~0):", train_scaled.mean(axis=0).round(4))
print("Training std (should be ~1):", train_scaled.std(axis=0).round(4))

Scaled shapes (Train / Test): (24000, 2) (6000, 2)
Training means (should be ~0): [ 0. -0.]
Training std (should be ~1): [1. 1.]


8. ColumnTransformer with Numeric & Categorical Pipelines
We construct a Scikit-Learn `ColumnTransformer` combining:
- A **numeric pipeline**: Median imputation (`SimpleImputer(strategy="median")`) followed by standard scaling (`StandardScaler()`).
- A **categorical pipeline**: Most-frequent imputation (`SimpleImputer(strategy="most_frequent")`) followed by One-Hot Encoding (`OneHotEncoder(handle_unknown="ignore")`).

The preprocessor is fitted strictly on `X_train` to ensure training-only parameter estimation.

In [12]:
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_encoder()),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("X_train_ready:", X_train_ready.shape)
print("X_test_ready:", X_test_ready.shape)

X_train_ready: (24000, 18)
X_test_ready: (6000, 18)


In [13]:
feature_names = preprocessor.get_feature_names_out()
print("number of output features:", len(feature_names))
print(feature_names.tolist())

number of output features: 18
['num__LIMIT_BAL', 'num__AGE', 'num__PAY_0', 'num__BILL_AMT1', 'num__PAY_AMT1', 'cat__SEX_1', 'cat__SEX_2', 'cat__EDUCATION_0', 'cat__EDUCATION_1', 'cat__EDUCATION_2', 'cat__EDUCATION_3', 'cat__EDUCATION_4', 'cat__EDUCATION_5', 'cat__EDUCATION_6', 'cat__MARRIAGE_0', 'cat__MARRIAGE_1', 'cat__MARRIAGE_2', 'cat__MARRIAGE_3']


### Interpretation & Data Leakage Prevention Explanation

**1. Output Inspection:**
- **Transformed Shapes**: `X_train_ready` has shape `(24000, 18)` and `X_test_ready` has shape `(6000, 18)`.
- **Feature Expansion**: The original 8 predictor columns (5 numeric + 3 categorical) expanded to 18 binary/scaled columns due to One-Hot Encoding of multi-class categorical variables (`SEX`, `EDUCATION`, `MARRIAGE`).

**2. How this workflow prevents Data Leakage:**
- **Training-Only Parameter Estimation**: All statistical parameters—specifically the numeric medians for imputation, the feature means ($\mu$) and standard deviations ($\sigma$) for standard scaling, and the unique categorical levels for one-hot encoding—are learned **strictly from `X_train`** via `.fit_transform()`.
- **Out-of-Sample Isolation**: `X_test` is processed using `.transform()` only. It uses the exact scaling constants and category mappings computed from `X_train`. The test set's own distribution remains completely unobserved during preprocessing, ensuring that evaluation metrics accurately reflect model generalization to unseen financial data.